# 📊 NumPy Essentials for AI

---

## 1. Overview

NumPy is the **foundation of the entire Python scientific computing stack**. PyTorch tensors are modeled after NumPy arrays. Understanding NumPy deeply means understanding how data flows through every AI system.

This notebook covers array creation, broadcasting, vectorized operations, and linear algebra — the NumPy skills used daily in AI engineering.

## 2. Learning Objectives

- Create arrays with various initialization strategies
- Master indexing, slicing, and boolean masking
- Understand and apply broadcasting rules
- Replace Python loops with vectorized NumPy operations
- Use NumPy's random module for reproducible experiments
- Apply linear algebra operations (matrix multiply, solve, SVD)

## 3. Imports

In [50]:
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"NumPy version: {np.__version__}")

NumPy version: 2.0.2


## 4. Configuration

In [51]:
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

## 5. Theory & Implementation

### 5.1 Array Creation & Dtypes

In [52]:
# Common array creation patterns in AI
print("=== Array Creation ===")

# From data
embedding = np.array([0.1, -0.3, 0.5, 0.8, -0.2], dtype=np.float32)
print(f"Embedding: {embedding}, dtype={embedding.dtype}")

# Zeros — bias initialization
bias = np.zeros(10, dtype=np.float32)
print(f"Bias (zeros): shape={bias.shape}")

# Ones — mask initialization
attention_mask = np.ones((4, 8), dtype=np.int32)
print(f"Attention mask (ones): shape={attention_mask.shape}")

# Identity — residual connections
identity = np.eye(4, dtype=np.float32)
print(f"Identity matrix:\n{identity}")

# Random normal — weight initialization (Xavier/He)
fan_in, fan_out = 768, 256
xavier_std = np.sqrt(2.0 / (fan_in + fan_out))
weights = np.random.randn(fan_out, fan_in).astype(np.float32) * xavier_std
print(f"\nXavier-init weights: shape={weights.shape}, std={weights.std():.4f} (target: {xavier_std:.4f})")

# Ranges — position indices
positions = np.arange(0, 512)  # Token positions
print(f"Position indices: {positions[:5]}...{positions[-3:]}")

# Linspace — temperature sweep
temperatures = np.linspace(0.1, 2.0, 10)
print(f"Temperature sweep: {temperatures}")

=== Array Creation ===
Embedding: [ 0.1 -0.3  0.5  0.8 -0.2], dtype=float32
Bias (zeros): shape=(10,)
Attention mask (ones): shape=(4, 8)
Identity matrix:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

Xavier-init weights: shape=(256, 768), std=0.0442 (target: 0.0442)
Position indices: [0 1 2 3 4]...[509 510 511]
Temperature sweep: [0.1    0.3111 0.5222 0.7333 0.9444 1.1556 1.3667 1.5778 1.7889 2.    ]


### 5.2 Indexing, Slicing & Boolean Masking

In [ ]:
# simulated model output: batch of 4 samples, 6 classes
logits = np.random.randn(4, 6)
class_names = ['cat', 'dog', 'bird', 'car', 'plane', 'ship']

print(f"Logits shape: {logits.shape}")
print(f"Logits:\n{logits}\n")

# basic indexing
print(f"First sample: {logits[0]}")
print(f"All samples, class 2: {logits[:, 2]}")
print(f"Samples 1-2, classes 0-2: \n{logits[1:3, :3]}\n")

# fancy indexing — get specific elements
predictions = np.argmax(logits, axis=1)  # Predicted class per sample
print(f"Predicted classes: {predictions}")
print(f"Predicted labels: {[class_names[i] for i in predictions]}")

# boolean masking — filter by condition
high_confidence = logits > 1.0
print(f"\nHigh confidence (> 1.0) count: {high_confidence.sum()}")
print(f"High confidence values: {logits[high_confidence]}")

# practical: mask padding tokens in attention
seq_lengths = np.array([5, 3, 7, 2])  # Actual lengths per sample
max_len = 8
mask = np.arange(max_len)[None, :] < seq_lengths[:, None]  # Broadcasting!
print(f"\nAttention mask (True = attend):\n{mask.astype(int)}")

Logits shape: (4, 6)
Logits:
[[-0.0036  0.463  -0.0361  1.4394  1.4131 -1.1515]
 [ 1.8218  0.5101 -0.8584 -1.3175 -0.5676  0.0292]
 [-1.2355 -1.0951  1.3603  0.8733 -0.7397 -0.975 ]
 [ 0.8986  1.7517 -1.1856  0.1809  0.0565  0.1955]]

First sample: [-0.0036  0.463  -0.0361  1.4394  1.4131 -1.1515]
All samples, class 2: [-0.0361 -0.8584  1.3603 -1.1856]
Samples 1-2, classes 0-2: 
[[ 1.8218  0.5101 -0.8584]
 [-1.2355 -1.0951  1.3603]]

Predicted classes: [3 0 2 1]
Predicted labels: ['car', 'cat', 'bird', 'dog']

High confidence (> 1.0) count: 5
High confidence values: [1.4394 1.4131 1.8218 1.3603 1.7517]

Attention mask (True = attend):
[[1 1 1 1 1 0 0 0]
 [1 1 1 0 0 0 0 0]
 [1 1 1 1 1 1 1 0]
 [1 1 0 0 0 0 0 0]]


### 5.3 Broadcasting — The Key to Vectorized Code

Broadcasting rules:
1. Arrays with fewer dimensions get dimensions **prepended** with size 1
2. Dimensions with size 1 are **stretched** to match the other array
3. If sizes don't match and neither is 1 → **error**

In [54]:
# Broadcasting examples

# 1) Scalar + Array (add bias to every element)
activations = np.random.randn(3, 4)
bias = 0.5
result = activations + bias  # Scalar broadcasts to (3, 4)
print(f"1) Scalar + Array: {activations.shape} + scalar → {result.shape}")

# 2) Row vector + Matrix (add per-feature bias)
X = np.random.randn(32, 768)  # Batch of 32, 768 features
bias_vec = np.random.randn(768)  # One bias per feature
result = X + bias_vec  # (32, 768) + (768,) → (32, 768)
print(f"2) Matrix + Row: {X.shape} + {bias_vec.shape} → {result.shape}")

# 3) Column vector + Row vector (outer product-like)
row = np.array([1, 2, 3])        # shape (3,)
col = np.array([[10], [20]])      # shape (2, 1)
result = row + col                 # (3,) → (1, 3), then (2, 1) + (1, 3) → (2, 3)
print(f"3) Col + Row: {col.shape} + {row.shape} → {result.shape}")
print(f"   Result:\n   {result}")

# 4) Practical: Z-score normalization (per feature)
data = np.random.randn(1000, 5) * np.array([1, 10, 100, 0.1, 50]) + np.array([0, 5, -3, 2, 10])
mean = data.mean(axis=0)     # (5,)
std = data.std(axis=0)       # (5,)
normalized = (data - mean) / std  # Broadcasting: (1000, 5) - (5,) / (5,)
print(f"\n4) Normalization: mean={normalized.mean(axis=0).round(2)}, std={normalized.std(axis=0).round(2)}")

1) Scalar + Array: (3, 4) + scalar → (3, 4)
2) Matrix + Row: (32, 768) + (768,) → (32, 768)
3) Col + Row: (2, 1) + (3,) → (2, 3)
   Result:
   [[11 12 13]
 [21 22 23]]

4) Normalization: mean=[-0. -0.  0. -0. -0.], std=[1. 1. 1. 1. 1.]


### 5.4 Vectorization — Why Loops Are Slow

In [55]:
# Speed comparison: Python loops vs. NumPy vectorization
n = 1_000_000
a = np.random.randn(n)
b = np.random.randn(n)

# Python loop
start = time.perf_counter()
result_loop = sum(a[i] * b[i] for i in range(n))
loop_time = time.perf_counter() - start

# NumPy vectorized
start = time.perf_counter()
result_numpy = np.dot(a, b)
numpy_time = time.perf_counter() - start

print(f"Python loop: {loop_time:.4f}s")
print(f"NumPy dot:   {numpy_time:.6f}s")
print(f"Speedup:     {loop_time / numpy_time:.0f}x")
print(f"Results match: {np.isclose(result_loop, result_numpy)}")
print(f"\n→ NumPy is ~{loop_time / numpy_time:.0f}x faster because it uses C/BLAS under the hood!")

Python loop: 0.6704s
NumPy dot:   0.003702s
Speedup:     181x
Results match: True

→ NumPy is ~181x faster because it uses C/BLAS under the hood!


In [56]:
# Common vectorized operations in AI

# 1) Softmax (vectorized)
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

batch_logits = np.random.randn(4, 6)
probs = softmax(batch_logits, axis=1)
print(f"Softmax output (rows sum to 1): {probs.sum(axis=1)}")

# 2) Cross-entropy loss (vectorized)
targets = np.array([0, 3, 2, 5])  # True class indices
# Gather probabilities for the true classes
target_probs = probs[np.arange(4), targets]
loss = -np.log(target_probs + 1e-8).mean()
print(f"Cross-entropy loss: {loss:.4f}")

# 3) Pairwise cosine similarity (vectorized)
embeddings = np.random.randn(5, 768)  # 5 embeddings, 768-d
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized = embeddings / norms
similarity_matrix = normalized @ normalized.T  # (5, 768) @ (768, 5) → (5, 5)
print(f"\nPairwise similarity matrix shape: {similarity_matrix.shape}")
print(f"Diagonal (self-similarity): {np.diag(similarity_matrix).round(4)}")

Softmax output (rows sum to 1): [1. 1. 1. 1.]
Cross-entropy loss: 1.6908

Pairwise similarity matrix shape: (5, 5)
Diagonal (self-similarity): [1. 1. 1. 1. 1.]


### 5.5 Reshaping & Axis Manipulation

Reshaping is used constantly in AI — for batching, attention heads, and tensor operations.

In [57]:
# Multi-head attention: split embedding into multiple heads
batch_size, seq_len, d_model = 2, 8, 64
n_heads = 4
d_head = d_model // n_heads  # 16

# Input: (batch, seq_len, d_model)
x = np.random.randn(batch_size, seq_len, d_model)
print(f"Input shape: {x.shape}")

# Reshape for multi-head attention: (batch, seq, d_model) → (batch, n_heads, seq, d_head)
x_heads = x.reshape(batch_size, seq_len, n_heads, d_head)  # (2, 8, 4, 16)
x_heads = x_heads.transpose(0, 2, 1, 3)                     # (2, 4, 8, 16)
print(f"Multi-head shape: {x_heads.shape}  →  (batch, n_heads, seq_len, d_head)")

# After attention, merge heads back
x_merged = x_heads.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, d_model)
print(f"Merged back: {x_merged.shape}")
print(f"Matches original shape: {x_merged.shape == x.shape}")

# Other common reshaping operations
print(f"\n--- Common Reshape Operations ---")
print(f"Flatten:     {x.reshape(batch_size, -1).shape}      (batch, seq*d_model)")
print(f"Add dim:     {x[:, :, np.newaxis, :].shape}  (batch, seq, 1, d_model)")
print(f"Squeeze:     {np.squeeze(x[:, :, np.newaxis, :], axis=2).shape}  (remove dim of size 1)")
print(f"Stack:       {np.stack([x, x, x]).shape}          (3, batch, seq, d_model)")
print(f"Concat:      {np.concatenate([x, x], axis=1).shape} (batch, 2*seq, d_model)")

Input shape: (2, 8, 64)
Multi-head shape: (2, 4, 8, 16)  →  (batch, n_heads, seq_len, d_head)
Merged back: (2, 8, 64)
Matches original shape: True

--- Common Reshape Operations ---
Flatten:     (2, 512)      (batch, seq*d_model)
Add dim:     (2, 8, 1, 64)  (batch, seq, 1, d_model)
Squeeze:     (2, 8, 64)  (remove dim of size 1)
Stack:       (3, 2, 8, 64)          (3, batch, seq, d_model)
Concat:      (2, 16, 64) (batch, 2*seq, d_model)


### 5.6 Random Number Generation — Reproducibility

In [58]:
# The modern way: use np.random.Generator (not the legacy np.random.seed)
rng = np.random.default_rng(seed=42)

# Common distributions used in AI
print("=== Random Distributions for AI ===")
print(f"Normal (weight init):   {rng.normal(0, 0.02, 5).round(4)}")
print(f"Uniform (dropout mask): {rng.uniform(0, 1, 5).round(4)}")
print(f"Integers (token IDs):   {rng.integers(0, 50000, 5)}")
print(f"Choice (sampling):      {rng.choice(['cat', 'dog', 'bird'], 3, replace=True)}")
print(f"Permutation (shuffle):  {rng.permutation(8)}")

# Reproducibility demo
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)
print(f"\nReproducible: {np.array_equal(rng1.normal(0, 1, 10), rng2.normal(0, 1, 10))}")

# Dropout mask
dropout_rate = 0.3
activations = rng.normal(0, 1, (3, 5))
mask = rng.uniform(0, 1, activations.shape) > dropout_rate
dropped = activations * mask / (1 - dropout_rate)  # Scale to maintain expected value
print(f"\nDropout mask:\n{mask.astype(int)}")
print(f"Dropped ~{(~mask).sum()}/{mask.size} activations ({(~mask).mean():.0%})")

=== Random Distributions for AI ===
Normal (weight init):   [ 0.0061 -0.0208  0.015   0.0188 -0.039 ]
Uniform (dropout mask): [0.9756 0.7611 0.7861 0.1281 0.4504]
Integers (token IDs):   [25017 18539  9127 46338 39078]
Choice (sampling):      ['dog' 'dog' 'bird']
Permutation (shuffle):  [1 0 4 7 5 6 2 3]

Reproducible: True

Dropout mask:
[[1 0 0 1 0]
 [1 1 1 1 1]
 [1 1 1 0 1]]
Dropped ~4/15 activations (27%)


## 7. Evaluation — Build a Simple Neural Network Forward Pass

In [59]:
# 2-layer neural network forward pass with pure NumPy
rng = np.random.default_rng(42)

# Network architecture: 784 → 128 → 10
input_dim, hidden_dim, output_dim = 784, 128, 10
batch_size = 32

# Initialize weights (He initialization)
W1 = rng.normal(0, np.sqrt(2 / input_dim), (input_dim, hidden_dim)).astype(np.float32)
b1 = np.zeros(hidden_dim, dtype=np.float32)
W2 = rng.normal(0, np.sqrt(2 / hidden_dim), (hidden_dim, output_dim)).astype(np.float32)
b2 = np.zeros(output_dim, dtype=np.float32)

# Fake input (like flattened 28x28 MNIST images)
X = rng.normal(0, 1, (batch_size, input_dim)).astype(np.float32)

# Forward pass
# Layer 1: linear + ReLU
z1 = X @ W1 + b1                          # (32, 784) @ (784, 128) + (128,) = (32, 128)
a1 = np.maximum(0, z1)                     # ReLU activation

# Layer 2: linear + softmax
z2 = a1 @ W2 + b2                          # (32, 128) @ (128, 10) + (10,) = (32, 10)
probs = softmax(z2, axis=1)                # (32, 10) — probability over 10 classes

print(f"Input:   {X.shape}")
print(f"Layer 1: {z1.shape} → ReLU → {a1.shape}")
print(f"Layer 2: {z2.shape} → Softmax → {probs.shape}")
print(f"\nPredicted classes: {np.argmax(probs, axis=1)[:10]}...")
print(f"Max probabilities: {np.max(probs, axis=1)[:10].round(3)}...")
print(f"Sum of probs per sample: {probs.sum(axis=1)[:5].round(4)} (should be 1.0)")
print(f"\nTotal parameters: {W1.size + b1.size + W2.size + b2.size:,}")
print("\n→ This is exactly what PyTorch's nn.Linear + nn.ReLU + nn.Softmax do!")

Input:   (32, 784)
Layer 1: (32, 128) → ReLU → (32, 128)
Layer 2: (32, 10) → Softmax → (32, 10)

Predicted classes: [3 3 3 6 4 6 4 1 3 6]...
Max probabilities: [0.198 0.217 0.513 0.195 0.608 0.312 0.366 0.375 0.249 0.225]...
Sum of probs per sample: [1. 1. 1. 1. 1.] (should be 1.0)

Total parameters: 101,770

→ This is exactly what PyTorch's nn.Linear + nn.ReLU + nn.Softmax do!


## 8. Exercises

### Exercise 1: Implement Layer Normalization
Layer norm normalizes across features (last dimension), used in every Transformer layer.

In [60]:
def layer_norm(x: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """Layer normalization: normalize across the last dimension."""
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

# Test
x_test = np.random.randn(2, 4, 8)  # (batch, seq, features)
x_normed = layer_norm(x_test)
print(f"Input mean per position: {x_test[0, 0].mean():.4f}")
print(f"Normed mean per position: {x_normed[0, 0].mean():.6f} (≈ 0)")
print(f"Normed std per position:  {x_normed[0, 0].std():.6f} (≈ 1)")

Input mean per position: 0.3820
Normed mean per position: -0.000000 (≈ 0)
Normed std per position:  0.999995 (≈ 1)


### Exercise 2: Implement Scaled Dot-Product Attention
The core attention formula: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

In [61]:
def scaled_dot_product_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray,
    mask: np.ndarray | None = None,
) -> np.ndarray:
    """Scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)  # (B, H, S, S)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ V  # (B, H, S, d_k)

# Test
B, H, S, D = 2, 4, 8, 16  # batch, heads, seq_len, d_head
Q = np.random.randn(B, H, S, D)
K = np.random.randn(B, H, S, D)
V = np.random.randn(B, H, S, D)
output = scaled_dot_product_attention(Q, K, V)
print(f"Attention output: {output.shape}  (batch, heads, seq_len, d_head)")
print("✅ Scaled dot-product attention — implemented from scratch!")

Attention output: (2, 4, 8, 16)  (batch, heads, seq_len, d_head)
✅ Scaled dot-product attention — implemented from scratch!


## 9. Challenge Problems

### Challenge 1: Implement Causal (Autoregressive) Attention Mask
Create a triangular mask so that each position can only attend to previous positions (used in GPT-style models). Apply it to the attention function above.

### Challenge 2: Implement Positional Encoding
Implement sinusoidal positional encoding from "Attention Is All You Need": $PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$, $PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$

### Challenge 3: Matrix Factorization Recommender
Implement a simple recommender system using matrix factorization (user-item matrix → U × V^T) trained with gradient descent.

## 10. Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [NumPy User Guide](https://numpy.org/doc/stable/user/) | 📖 Docs | Official comprehensive guide |
| [From Python to NumPy](https://www.labri.fr/perso/nrougier/from-python-to-numpy/) | 📘 Book | Free, excellent deep dive |
| [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) | 📖 Docs | Official broadcasting rules |
| [Jay Alammar — Visual NumPy](https://jalammar.github.io/visual-numpy/) | 📝 Blog | Visual guide to array operations |

---

**Next:** [06 — Pandas & Polars →](06-pandas-polars-data.ipynb)